# exp102 confidence gated likPF fallback on exp101

exp101 の saved booster と exp099 v2 cache を使い、`likpf_mean` default の高信頼 row 限定 fallback を train-side OOF で監査する。

## Contents

1. Setup and configuration
2. Input and model checks
3. Reconstruct exp101 scores and evaluate gates
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from confidence_gated_likpf_fallback_on_exp101 import run_confidence_gated_likpf_fallback

paths = ExperimentPaths()
config = load_config()
print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('cache parent:', get_nested(config, 'lineage.cache_parent'))
print('default candidate:', get_nested(config, 'gate.default_candidate'))
print('allowed switch candidates:', get_nested(config, 'gate.allowed_switch_candidates'))
print('output dir:', paths.artifacts_dir)


## 2. Input and model checks

In [ ]:
candidate_files = [
    Path(get_nested(config, 'data.exp099_train_feature_cache_local')),
    Path(get_nested(config, 'data.exp099_train_feature_schema_local')),
    Path(get_nested(config, 'data.exp101_artifact_dir_local')) / get_nested(config, 'data.exp101_model_manifest'),
    Path(get_nested(config, 'data.exp101_artifact_dir_local')) / get_nested(config, 'data.exp101_feature_schema'),
]
for path in candidate_files:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print('kernel sources:', get_nested(config, 'runtime.kaggle.kernel_sources'))


## 3. Reconstruct exp101 scores and evaluate gates

In [ ]:
summary = run_confidence_gated_likpf_fallback(
    output_dir=Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else paths.artifacts_dir,
    cache_path=None,
    schema_path=None,
    max_rows=None,
)
print(json.dumps(summary['decision'], indent=2, sort_keys=True))


## 4. Metrics and artifacts

In [ ]:
artifact_dir = Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else paths.artifacts_dir
metrics_path = artifact_dir / 'exp102_confidence_gated_likpf_fallback_on_exp101_metrics.csv'
by_well_path = artifact_dir / 'exp102_confidence_gated_likpf_fallback_on_exp101_by_well.csv'
bucket_path = artifact_dir / 'exp102_confidence_gated_likpf_fallback_on_exp101_bucket_metrics.csv'
summary_path = artifact_dir / 'exp102_confidence_gated_likpf_fallback_on_exp101_summary.json'

metrics = pd.read_csv(metrics_path)
display(metrics.head(20))
display(metrics[metrics['mode'].eq('gated')].sort_values('rmse_tvt').head(10))
display(pd.read_csv(by_well_path).sort_values('rmse_tvt', ascending=False).head(20))
display(pd.read_csv(bucket_path).head(20))
print(summary_path)
print(json.dumps(json.loads(summary_path.read_text())['sha256'], indent=2, sort_keys=True))
